In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from utility import orbit, BrusselatorModel, optim_BrusselatorModel
import sys, pickle, os, datetime,time
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
#Load pickle file from a directory
# bruss_dflt_dir = "/home/aziz-adm/Bureau/PhD/code/num_solutions/Orbit-Solutions-/Results/bruss_dflt_params.in/2025-07-30_17"
bruss_dflt_dir = "/home/aziz-adm/Bureau/PhD/code/num_solutions/Orbit-Solutions-/Results/RK45_bruss_dflt_params.in/2025-08-31"
notfull_subiter_dir = "/home/aziz-adm/Bureau/PhD/code/num_solutions/Orbit-Solutions-/Results/RK45_notfull_subiter_bruss.in/2025-08-31"
# p_equal_2nz_dir = "/home/aziz-adm/Bureau/PhD/code/num_solutions/Orbit-Solutions-/Results/RK45_p_equal_2nz_bruss.in/2025-08-31"
# #Go through the directory and find the pickle files
# p_equal_2nz_files = [f for f in os.listdir(p_equal_2nz_dir) if f.endswith('.pkl')]
bruss_dflt_files = [f for f in os.listdir(bruss_dflt_dir) if f.endswith('.pkl')]
# bruss_dflt_dir_files2 = [f for f in os.listdir(bruss_dflt_dir2) if f.endswith('.pkl')]
notfull_subiter_files = [f for f in os.listdir(notfull_subiter_dir) if f.endswith('.pkl')]

#Load the pickle files
# Initialize an empty DataFrame to concatenate results
DATA=[]
for dir in [bruss_dflt_dir, notfull_subiter_dir]:#, p_equal_2nz_dir]:
    df = pd.DataFrame()
    for file in [f for f in os.listdir(dir) if f.endswith('.pkl')]:
        with open(os.path.join(dir, file), 'rb') as f:
            data = pickle.load(f)
            df = pd.concat([df,pd.DataFrame(data[1], index=[0])])  # Assuming data[1] is a dictionary or similar structure
    DATA.append(df)
#concatanate the data into a DataFrame
df.sort_values(by='orbit_method', inplace=True)  # Sort by 'nz' column

In [ ]:
df0 = DATA[0]
df0

In [ ]:
D = df0.groupby(['orbit_method', 'nz']).mean().reset_index()

#keep the scientific notation format for the values in the columns n_iter
D

In [ ]:
#plot comput_time vs nz only for Newton method
fontsize = 20
plt.figure(figsize=(10, 6), dpi=300)
Newton = df0[df0['orbit_method'] == 'Newton_orbit'].reset_index()
Newton.sort_values(by='nz', inplace=True)  # Sort by 'nz' column
plt.plot(Newton['nz'], Newton['comput_time'], marker='o', label='Newton')
for k, df in enumerate(DATA):
    # for method in df0['orbit_method'].unique():
    subset = df[df['orbit_method'] == 'Newton_Picard_sub_proj'].reset_index()
    subset.sort_values(by='nz', inplace=True)  # Sort by 'nz' column
    #get p0 and sub_sp_iter from the first row of subset
    p0 = subset['p0'].iloc[0]
    nu_sub = subset['sub_sp_iter'].iloc[0]
    # print(p0)
    if k== 0:
        label = r'NP with full subspace iteration, $\nu_{sub}$=%i, $p_0$=%i' % (nu_sub, p0)
    elif k==1:
        label = r"NP with not full subspace iteration, $p_0$=5"   
    else:
        label = r'NP with, $\nu_{sub}$= %i, $p_0$=dim' %nu_sub
    plt.plot(subset['nz'], subset['comput_time'], marker='o', label= label) if k!=0 else None
    plt.xlabel('nz', fontsize=fontsize)
    plt.ylabel('Computational Time (s)', fontsize=fontsize)
    plt.title('Computational Time vs Grid size',   fontsize=fontsize)
    plt.yscale('log')  # Set y-axis to logarithmic scale for better visibility
    # plt.xscale('log')  # Set x-axis to logarithmic scale for better visibility
    
    plt.legend(fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    # Save the plot
plt.savefig('comput_time_vs_nz_newton_NP_notfull_subiter.png')
plt.show()

In [ ]:
df0

In [ ]:
#Plot bar precision vs nz for different orbit methods
fontsize = 20
plt.figure(figsize=(10, 6), dpi=300)
Newton = df0[df0['orbit_method'] == 'Newton_orbit'].reset_index()
Newton.sort_values(by='nz', inplace=True)  # Sort by 'nz' column
plt.plot(Newton['nz'], Newton['n_iter'],'o--', label='Newton', alpha=0.7)
for k, df in enumerate(DATA):
    # for method in df0['orbit_method'].unique():
    subset = df[df['orbit_method'] == 'Newton_Picard_sub_proj'].reset_index()
    subset.sort_values(by='nz', inplace=True)  # Sort by 'nz' column
    #get p0 and sub_sp_iter from the first row of subset
    p0 = subset['p0'].iloc[0]
    nu_sub = subset['sub_sp_iter'].iloc[0]
    print(p0)
    if k== 0:
        label = r'NP with full subspace iteration, $\nu_{sub}$=%i, $p_0$=%i' % (nu_sub, p0)
    elif k==1:
        label = r"NP with not full subspace iteration, $p_0$=5"   
    else:
        label = r'NP with, $\nu_{sub}$= %i, $p_0$=dim' %nu_sub
    plt.plot(subset['nz'], subset['n_iter'],'o--', label=label, alpha=1)
    plt.xlabel('nz', fontsize=fontsize)
    plt.ylabel('Iteration', fontsize=fontsize)
    plt.title('Iteration vs nz', fontsize=fontsize)
    plt.ylim(1,14)
    # plt.yscale('log')  # Set y-axis to logarithmic scale for better visibility
    # plt.xscale('log')  # Set x-axis to logarithmic scale for
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    # Save the plot
plt.savefig('iteration_to_reach_convergence_vs_nz.png')
plt.show()

In [ ]:
#A python script to merge pdf files
import os
from PyPDF2 import PdfMerger

#Files to be merged
file1 = '/home/aziz-adm/Bureau/PhD/Administratives/CSI/rapport_csi_2024_2025.pdf'
file2 = '/home/aziz-adm/Bureau/PhD/Administratives/CSI/DED_Diallo.pdf'

pdf_files = [file1, file2]
# Output merged PDF file
output_pdf = "dossier_inscription_2025_2026.pdf"
# Create a PdfMerger object
merger = PdfMerger()
# Loop through the PDF files and append them to the merger
for pdf in pdf_files:
    merger.append(pdf)
# Write out the merged PDF to a file
merger.write(output_pdf)



In [ ]:
# 2D TORUS using plotly
import plotly.graph_objects as go

theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, 2*np.pi, 100)
Theta, Phi = np.meshgrid(theta, phi)
R = 5
r = 2.0
X = (R + r * np.cos(Theta) )* np.cos(Phi)
Y = (R + r * np.cos(Theta) )* np.sin(Phi)
Z = r * np.sin(Theta)
fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', opacity=0.6)])
fig.add_scatter3d(x=R * np.sin(z_centers) * np.sin(z_centers), y=R * np.sin(z_centers) * np.sin(z_centers), z=R * np.cos(z_centers)*np.sin(z_centers), mode='markers', name='Mesh Centers', marker=dict(color='red', size=5))
fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.show()

# theta = np.linspace(0, 2*np.pi, 100)
# phi = np.linspace(0, 2*np.pi, 100)
# Theta, Phi = np.meshgrid(theta, phi)
# R = 1
# X = R * np.cos(Theta) * np.cos(Phi)
# Y = R * np.cos(Theta) * np.sin(Phi)
# Z = R * np.sin(Theta)
# fig = plt.figure(figsize=(8, 6))
# ax = fig.add_subplot(111, projection='3d')
# ax.plot_surface(X, Y, Z, color='c', alpha=0.6)
# ax.scatter(R * np.cos(z_centers) * np.cos(z_centers), R * np.cos(z_centers) * np.sin(z_centers), R * np.sin(z_centers), c='r', label='Mesh Centers')
# ax.set_xlabel('X')
# ax.set_ylabel('Y')
# ax.set_zlabel('Z')
# ax.set_title('2D Torus with Mesh Centers')
# ax.legend()
# plt.show()